In [ ]:
import asyncio
import socket

async def receive_number_coroutine(reader):
    data = await reader.read(100)
    message = data.decode()
    addr = reader._transport.get_extra_info('peername')
    print(f"Received {message} from {addr}")
    return message

async def main(host, port):
    # Connect to the server
    reader, writer = await asyncio.open_connection(host, port)

    try:
        # Receive number asynchronously
        number = await receive_number_coroutine(reader)

        # Check if the data received is a number within the range
        try:
            number = int(number)
            if 0 <= number <= 10:
                print("Received number within range: ", number)
            else:
                print("The number is not in the range 0-10.")
        except ValueError:
            print("Received data is not a valid number.")

    except Exception as e:
        print("An error occurred: ", e)
    
    finally:
        # Close the connection
        writer.close()
        await writer.wait_closed()

# Replace 'SERVER_IP_ADDRESS' with the actual IP address of the server Raspberry Pi
server_ip = '10.243.67.147' # Example: '192.168.1.2'
port = 5560

# Run the main function until it is complete
asyncio.run(main(server_ip, port))

# ~~~~~~~~~~~~
# Drafts / Old


In [ ]:
import socket
import time
from IPython.display import clear_output


# Set the host and port for the server (Robot 1)
host = '10.243.67.147'# Replace with the IP address of Robot 2
port = 5560

s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
s.connect((host, port))

start_time = time.time()
number = 0

while time.time() - start_time < 20:
    print("\n\n")
    print("Request Button Value")
    #ASK FOR A BUTTON VALUE
    s.send(str.encode('GET_BUTTON'))
    reply = s.recv(1024)
    print("Number State:", reply.decode('utf-8'))
    
    #SEND A MESSAGE
    number = number + 1
    s.send(str.encode('SEND '+str(number)))
    reply = s.recv(1024)
    #print(reply.decode('utf-8'))
    print("Sending:", number)
    
    if number%5==0:
        #every tenth loop clear the display
        clear_output()
           
s.send(str.encode("KILL"))          
s.close()
clear_output()
print('DONE')


# TESTING

In [ ]:
import socket
import time

HOST = '10.243.76.73'# Replace with the IP address of Robot 2
PORT = 5560  # The port used by the server

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
    # Connect to the server
    sock.connect((HOST, PORT))

    # Receive data from the server
    data = sock.recv(1024).decode("utf-8")
    number = int(data)

    # Print the received number
    print(f"Received number: {number}")


In [ ]:
import socket
import time
import tkinter as tk

HOST = '10.243.76.73'# Replace with the IP address of Robot 2
PORT = 5560  # The port used by the server

def update_bar():
    # Receive data from the server
    sock.setblocking(True)
    data = sock.recv(1024).decode("utf-8")
    number = int(data)

    # Update the bar chart based on the received number
    for i in range(number):
        bar_widgets[i]['fill'] = "green"

    root.update()

    # Reset the bar chart after a delay
    time.sleep(1)
    for i in range(10):
        bar_widgets[i]['fill'] = "gray"

root = tk.Tk()
root.title("Bar Chart")
root.configure(background="black")

# Create a frame to hold the bar chart
bar_frame = tk.Frame(root, background="black")
bar_frame.pack()

# Create 10 bar widgets
bar_widgets = []
for i in range(10):
    bar = tk.Label(bar_frame, width=100, height=20, bg="gray")
    bar.pack()
    bar_widgets.append(bar)

# Connect to the server
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
    # Connect to the server
    sock.connect((HOST, PORT))

    # Start updating the bar chart
    update_bar()

root.mainloop()
